In [ ]:
# ── Cell 1: Mount Drive and set working directory ─────────────────────────
from google.colab import drive  # type: ignore
import os, subprocess, sys
from pathlib import Path

drive.mount("/content/drive")

# ── Locate project root — check every common Drive layout ─────────────────
_CANDIDATES = [
    "/content/drive/MyDrive/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant",
    "/content/drive/MyDrive/smart-learning-assistant/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant",
    "/content/drive/MyDrive/PixelLab-StudyPal-RAG-DIP",
    "/content/drive/MyDrive/smart-learning-assistant",
]

DRIVE_PROJECT_ROOT = next(
    (p for p in _CANDIDATES
     if Path(p).is_dir() and Path(p, "app").is_dir()),  # must have app/ subdir
    None,
)

if DRIVE_PROJECT_ROOT is None:
    # Last-resort: ask user
    print("⚠️  Could not auto-detect project folder.")
    print("    Available top-level Drive folders:")
    for d in sorted(Path("/content/drive/MyDrive").iterdir()):
        if d.is_dir():
            print("       ", d)
    DRIVE_PROJECT_ROOT = input(
        "\nEnter FULL path to the smart-learning-assistant folder: "
    ).strip()
    if not Path(DRIVE_PROJECT_ROOT, "app").is_dir():
        sys.exit("❌ Path not valid — no app/ subdirectory found. Fix the path and re-run.")

print(f"Project root : {DRIVE_PROJECT_ROOT}")

# ── Pull latest code ───────────────────────────────────────────────────────
_repo_root = str(Path(DRIVE_PROJECT_ROOT).parent)
_pull = subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=_repo_root, capture_output=True, text=True,
)
_msg = (_pull.stdout + _pull.stderr).strip() or "Already up to date."
print("git pull:", _msg)

os.chdir(DRIVE_PROJECT_ROOT)
print(f"Working dir  : {os.getcwd()}")
print("Files        :", sorted(os.listdir(".")))


In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
# ragas 0.1.x used langchain_core.pydantic_v1, removed in langchain-core 0.3.
# That forced numpy→1.26.4, which breaks Colab's pandas (compiled for numpy 2).
# ragas>=0.2.0 dropped pydantic_v1 → works with langchain-core 0.3.x + numpy 2.x.
%pip install -q \
    "ragas>=0.2.0,<0.3.0" \
    "langchain-groq>=0.2.0" \
    "langchain-community>=0.3.0" \
    "langchain-huggingface>=0.1.0" \
    "datasets>=2.18" \
    "sentence-transformers" \
    "python-dotenv"

print("\n✅ Packages installed. Restart runtime if Colab prompts you.")


In [ ]:
# ── Cell 3: Set Groq API key (free judge LLM for RAGAS) ───────────────────
# Get a free key at https://console.groq.com/keys (sign in with GitHub/Google)
import os
from getpass import getpass

groq_key = getpass("Enter your Groq API key (gsk_...): ")
os.environ["GROQ_API_KEY"] = groq_key

# Quick sanity check
from langchain_groq import ChatGroq  # type: ignore
_test = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
_resp = _test.invoke("Reply with the single word OK.")
_text = _resp.content if isinstance(_resp.content, str) else (_resp.content[0] if _resp.content else "OK")
print(f"✅ Groq key valid. Test response: {_text}")
print("   RAGAS judge  : llama-3.1-8b-instant (500K TPD)")
print("   Embeddings   : all-MiniLM-L6-v2 (local, sentence-transformers)")
print("   RunConfig    : max_workers=1 (sequential) · timeout=3600s")


In [ ]:
# ── Cell 4: Load intermediate JSON ────────────────────────────────────────
import json
from pathlib import Path

INTERMEDIATE_PATH = Path("data/eval_intermediate.json")

if not INTERMEDIATE_PATH.exists():
    # Fallback: allow manual upload from local machine
    from google.colab import files # type: ignore
    print("eval_intermediate.json not found in Drive project folder.")
    print("Uploading from your local machine instead...")
    uploaded = files.upload()          # shows a file-picker in Colab
    INTERMEDIATE_PATH = Path(list(uploaded.keys())[0])

with open(INTERMEDIATE_PATH, encoding="utf-8") as f:
    intermediate = json.load(f)

n_questions  = len(intermediate["questions"])
n_guardrail  = len(intermediate.get("guardrail_results", []))
collected_at = intermediate.get("collected_at", "unknown")

print(f"✅ Loaded intermediate data")
print(f"   DIP questions  : {n_questions}")
print(f"   Guardrail tests: {n_guardrail}")
print(f"   Collected at   : {collected_at}")

# Preview first question
print(f"\n   Q1: {intermediate['questions'][0]}")
print(f"   A1: {intermediate['answers'][0][:120]}...")

In [ ]:
# ── Cell 5: RAGAS scoring ─────────────────────────────────────────────────
# ragas 0.2.x column names: user_input / response / retrieved_contexts / reference
#
# Token-budget strategy (Groq free tier: 500K TPD):
#   - max_tokens=1024 on judge LLM: RAGAS JSON responses are 50–400 tokens,
#     so 1024 is plenty. The old 4096 caused preamble-heavy responses that
#     burned ~166K tokens/metric × 4 = 664K > 500K daily limit.
#   - InMemoryRateLimiter(0.1 req/s): stays under 6000 TPM ceiling.
#   - Checkpoint after every metric: if quota is ever hit, the next run
#     skips already-scored metrics and only runs the remaining ones.
import sys, os, time, warnings, json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")
sys.path.insert(0, str(Path.cwd()))

from datasets import Dataset
from ragas import evaluate
try:
    from ragas import RunConfig
except ImportError:
    from ragas.run_config import RunConfig

from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Rate limiter ───────────────────────────────────────────────────────────
_rate_limiter   = None
_between_metric = 30        # seconds to sleep between metrics (TPM window flush)

try:
    from langchain_core.rate_limiters import InMemoryRateLimiter
    _rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.1,
        check_every_n_seconds=0.05,
        max_bucket_size=1,
    )
    print("✅ Rate limiter: 0.1 req/s (every LLM sub-call throttled automatically)")
except ImportError:
    _between_metric = 90
    print("⚠️  InMemoryRateLimiter unavailable — sleeping 90s between metrics.")

# ── Preamble-stripping wrapper ─────────────────────────────────────────────
# llama-3.1-8b-instant sometimes emits chain-of-thought BEFORE the JSON block.
# We strip it in-place at the ChatGroq level before ragas ever sees the text.
class _CleanChatGroq(ChatGroq):
    @staticmethod
    def _strip(text: str) -> str:
        idx = text.find('{')
        return text[idx:] if idx > 0 else text

    def _clean_result(self, result):
        for gen_list in result.generations:
            for gen in gen_list:
                text = getattr(gen, 'text', None)
                if not isinstance(text, str):
                    continue
                clean = self._strip(text)
                if clean == text:
                    continue
                try:
                    gen.text = clean
                except Exception:
                    try:
                        object.__setattr__(gen, 'text', clean)
                    except Exception:
                        pass
        return result

    def generate_prompt(self, prompts, stop=None, callbacks=None, **kwargs):
        return self._clean_result(
            super().generate_prompt(prompts, stop=stop, callbacks=callbacks, **kwargs))

    async def agenerate_prompt(self, prompts, stop=None, callbacks=None, **kwargs):
        return self._clean_result(
            await super().agenerate_prompt(prompts, stop=stop, callbacks=callbacks, **kwargs))

# ── Judge LLM + embeddings ─────────────────────────────────────────────────
# max_tokens=1024: RAGAS only generates short JSON (50-400 tokens).
# The previous 4096 let the model generate long preamble, burning daily quota.
_groq_kwargs = dict(model="llama-3.1-8b-instant", temperature=0, max_tokens=1024)
if _rate_limiter is not None:
    _groq_kwargs["rate_limiter"] = _rate_limiter

_raw_llm  = _CleanChatGroq(**_groq_kwargs)
ragas_llm = LangchainLLMWrapper(_raw_llm)
ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

_METRICS     = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
_TARGET      = 0.7
_metric_objs = [faithfulness, answer_relevancy, context_precision, context_recall]
for _m in _metric_objs:
    _m.llm = ragas_llm
    if hasattr(_m, "embeddings"):
        _m.embeddings = ragas_emb

# ── Load intermediate JSON (git pull already ran in Cell 1) ───────────────
_inter_path = Path("data/eval_intermediate.json")
if not _inter_path.exists():
    from google.colab import files
    print("eval_intermediate.json not found — upload it now:")
    uploaded = files.upload()
    _inter_path = Path(list(uploaded.keys())[0])

with open(_inter_path, encoding="utf-8") as _f:
    intermediate = json.load(_f)

_n = len(intermediate["questions"])
print(f"✅ Loaded {_n} questions  (collected: {intermediate.get('collected_at','?')})")

# ── Extract Formal Concept section for faithfulness evaluation ─────────────
# The system prompt structures every answer as:
#   ### 💡 The Intuition         → real-world metaphor (NOT in PDF chunks)
#   ### 📖 The Formal Concept    → academic definition (grounded in retrieved context)
#   ### 💻 Python Implementation → LLM-generated code  (NOT in PDF chunks)
#   ### 🎓 Mentor's Note         → mixed knowledge
#
# We evaluate ONLY the Formal Concept section (📖 → 💻 boundary) so that
# RAGAS faithfulness measures only textbook-grounded claims.
def _extract_formal(answer: str) -> str:
    START_MARKERS = ("### 📖 The Formal Concept", "### 📖 Formal Concept", "📖")
    STOP_MARKERS  = (
        "### 💻 Python Implementation", "### 💻", "💻 Python",
        "### 🎓 Mentor", "🎓",
    )
    start_idx = -1
    for marker in START_MARKERS:
        idx = answer.find(marker)
        if idx >= 0:
            start_idx = idx
            break
    if start_idx < 0:
        return answer[:2000]
    end_idx = len(answer)
    for marker in STOP_MARKERS:
        idx = answer.find(marker, start_idx + 10)
        if 0 < idx < end_idx:
            end_idx = idx
    return answer[start_idx:end_idx].strip()[:3000]

_answers_for_eval = [_extract_formal(a) for a in intermediate["answers"]]
_formal_found = sum(1 for a in intermediate["answers"] if "📖" in a)
_py_excluded  = sum(1 for a in intermediate["answers"] if "💻" in a)
print(f"Formal section extraction: {_formal_found}/{_n} had '📖' marker")
print(f"Python code excluded:      {_py_excluded}/{_n} answers had '💻' section stripped")

# ── Build contexts and reference ───────────────────────────────────────────
_raw_ctxs = intermediate.get("contexts", [])
_sample   = _raw_ctxs[0][0] if _raw_ctxs and _raw_ctxs[0] else ""
_is_text  = len(_sample) > 100 and not _sample.endswith(".pdf")

if _is_text:
    _contexts_for_eval = _raw_ctxs
    _ctx_count = len(_raw_ctxs[0]) if _raw_ctxs else 0
    print(f"✅ Real page_content contexts — {_ctx_count} chunks/question, {len(_sample)} chars in first chunk")
else:
    print(f"⚠️  Contexts look like filenames ('{_sample[:60]}') — using ground_truths as fallback")
    _gt_raw  = intermediate.get("ground_truths", [])
    _gt_flat = [(g[0] if isinstance(g, list) else str(g)) for g in _gt_raw][:_n]
    _contexts_for_eval = [[g] for g in _gt_flat]

_gt_raw    = intermediate.get("ground_truths", [])
_reference = [(g[0] if isinstance(g, list) else str(g)) for g in _gt_raw][:_n]
if len(_reference) < _n:
    _reference += _answers_for_eval[len(_reference):]

print(f"Reference: {len(_reference)} ground-truth strings loaded")

dataset = Dataset.from_dict({
    "user_input":         intermediate["questions"],
    "response":           _answers_for_eval,
    "retrieved_contexts": _contexts_for_eval,
    "reference":          _reference,
})

# ── Checkpoint: resume across daily quota resets ───────────────────────────
# Scores are saved to disk after every metric. If the 4th metric fails due to
# daily quota exhaustion, simply re-run Cell 5 the next day with a fresh key —
# the first three metrics load instantly from checkpoint and only the missing
# one is scored against the LLM. Delete the file to force a full fresh run.
_CHECKPOINT_PATH = Path("data/ragas_checkpoint.json")
_checkpoint: dict = {}

if _CHECKPOINT_PATH.exists():
    try:
        with open(_CHECKPOINT_PATH, encoding="utf-8") as _cf:
            _checkpoint = json.load(_cf)
        _done = [m for m in _METRICS if m in _checkpoint]
        _todo = [m for m in _METRICS if m not in _checkpoint]
        if _done:
            print(f"\n♻️  Checkpoint found → skipping: {_done}")
            print(f"   Will score:          {_todo}")
            print("   (Delete data/ragas_checkpoint.json to force a full re-run)\n")
    except Exception as _ckpt_err:
        print(f"⚠️  Checkpoint unreadable ({_ckpt_err}) — starting fresh.")
        _checkpoint = {}

# ── Run one metric at a time ───────────────────────────────────────────────
run_cfg = RunConfig(timeout=3600, max_workers=1)

_remaining = [m for m in _METRICS if m not in _checkpoint]
print(f"Running {len(_remaining)}/4 metrics × {_n} questions  (rate-limited · sequential)")
if _remaining:
    print(f"⏱  Expected: ~{len(_remaining)*12}–{len(_remaining)*15} min.  Do NOT interrupt!\n")

_score_columns: dict = {}

for _i, (_mobj, _mname) in enumerate(zip(_metric_objs, _METRICS)):

    # ── Load from checkpoint if already computed ──────────────────────────
    if _mname in _checkpoint:
        _score_columns[_mname] = _checkpoint[_mname]
        _avg = float(np.nanmean(_checkpoint[_mname]))
        print(f"[{_i+1}/4] {_mname:25s} ♻️  from checkpoint   mean={_avg:.3f}  "
              f"{'✅' if _avg >= _TARGET else '❌'}")
        continue

    # ── Score this metric ─────────────────────────────────────────────────
    _t0 = time.time()
    print(f"[{_i+1}/4] {_mname} …", flush=True)

    _r    = evaluate(dataset, metrics=[_mobj], llm=ragas_llm,
                     embeddings=ragas_emb, run_config=run_cfg)
    _df_m = _r.to_pandas() if hasattr(_r, "to_pandas") else pd.DataFrame(dict(_r))
    _elapsed = time.time() - _t0

    if _mname in _df_m.columns:
        _score_columns[_mname] = _df_m[_mname].tolist()
        _avg = float(_df_m[_mname].mean())
    else:
        _numeric = [c for c in _df_m.columns
                    if c not in ("user_input","response","retrieved_contexts","reference")]
        if _numeric:
            _score_columns[_mname] = _df_m[_numeric[0]].tolist()
            _avg = float(_df_m[_numeric[0]].mean())
        else:
            _score_columns[_mname] = [float("nan")] * len(dataset)
            _avg = float("nan")

    print(f"    {'✅' if _avg >= _TARGET else '❌'} mean={_avg:.3f}  ({_elapsed/60:.1f} min)")

    # ── Save checkpoint immediately after this metric ─────────────────────
    _checkpoint[_mname] = _score_columns[_mname]
    _CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(_CHECKPOINT_PATH, "w", encoding="utf-8") as _cf:
        json.dump(_checkpoint, _cf)
    print(f"   💾 Checkpoint saved  ({list(_checkpoint.keys())})")

    if _i < 3 and (_METRICS[_i + 1] not in _checkpoint):
        print(f"   ↳ sleeping {_between_metric}s …")
        time.sleep(_between_metric)

# ── Clear checkpoint — all 4 metrics complete ─────────────────────────────
if all(m in _checkpoint for m in _METRICS):
    if _CHECKPOINT_PATH.exists():
        _CHECKPOINT_PATH.unlink()
    print("\n🗑️  Checkpoint cleared — all 4 metrics complete.")

# ── Results ────────────────────────────────────────────────────────────────
ragas_df = pd.DataFrame({
    "user_input": dataset["user_input"],
    "response":   dataset["response"],
    **_score_columns,
})

_csv_path = Path("data/ragas_scores.csv")
_csv_path.parent.mkdir(parents=True, exist_ok=True)
ragas_df.to_csv(_csv_path, index=False)
print(f"\n✅ Scores saved → {_csv_path}")

print("\n=== RAGAS Scores ===")
print(f"{'Metric':<25} {'Score':>7}  {'Target':>7}  Status")
print("-" * 52)
for m in _METRICS:
    if m in ragas_df.columns:
        score  = float(ragas_df[m].mean())
        status = "✅ PASS" if score >= _TARGET else "❌ FAIL"
        print(f"{m:<25} {score:>7.3f}  {_TARGET:>7.1f}  {status}")

# ── Markdown report ────────────────────────────────────────────────────────
def _build_report(df, latencies, guardrail_results, topic_map=None):
    ts   = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    lats = sorted(latencies) if latencies else []
    lines = [
        "# Evaluation Report — DIP AI Tutor",
        f"\n_Generated: {ts}_\n",
        "\n## Overall RAGAS Scores\n",
        "| Metric | Score | Target | Status |",
        "|--------|-------|--------|--------|",
    ]
    for m in _METRICS:
        if m in df.columns:
            s = float(df[m].mean())
            lines.append(f"| {m} | {s:.3f} | {_TARGET:.1f} | {'✅' if s >= _TARGET else '❌'} |")

    if topic_map and all(c in df.columns for c in ["faithfulness", "answer_relevancy"]):
        _df2 = df.copy()
        _df2["topic"] = topic_map[:len(_df2)]
        lines += ["\n## Per-Topic Breakdown\n",
                  "| Topic | Faithfulness | Answer Relevancy |",
                  "|-------|-------------|-----------------|"]
        for topic, grp in _df2.groupby("topic"):
            lines.append(f"| {str(topic)[:60]} | {float(grp['faithfulness'].mean()):.3f} "
                         f"| {float(grp['answer_relevancy'].mean()):.3f} |")

    if lats:
        p95 = float(np.percentile(lats, 95))
        lines += ["\n## Latency Analysis\n",
                  f"- **Mean**: {float(np.mean(lats)):.2f}s",
                  f"- **p50**: {float(np.percentile(lats, 50)):.2f}s",
                  f"- **p95**: {p95:.2f}s" + (" ⚠️ exceeds 5 s target" if p95 > 5 else " ✅")]

    if guardrail_results:
        lines += ["\n## Guardrail Test Results\n",
                  "| Question | Status | Answer Preview |",
                  "|----------|--------|----------------|"]
        for g in guardrail_results:
            lines.append(f"| {str(g.get('question',''))[:60]} "
                         f"| {g.get('status','?')} "
                         f"| {g.get('answer','')[:80].replace(chr(10),' ')} |")

    lines += ["\n## Failed Cases (score < 0.7)\n"]
    any_bad = False
    for idx, (_, row) in enumerate(df.iterrows()):
        bad = [m for m in _METRICS if m in row and float(row[m]) < _TARGET]
        if bad:
            any_bad = True
            lines.append(f"**Q{idx+1}**: {str(row.get('user_input',''))[:80]}")
            for m in bad:
                lines.append(f"  - `{m}`: {float(row[m]):.3f}")
    if not any_bad:
        lines.append("_All questions meet the 0.7 threshold._ ✅")

    return "\n".join(lines)

report_md = _build_report(
    ragas_df,
    latencies=intermediate.get("latencies", []),
    guardrail_results=intermediate.get("guardrail_results", []),
    topic_map=intermediate.get("topic_map"),
)
Path("evaluation_report.md").write_text(report_md, encoding="utf-8")
print("✅ Report saved → evaluation_report.md")


In [ ]:
# ── Cell 6: Save report to Drive and offer local download ─────────────────
from pathlib import Path
from google.colab import files as colab_files # type: ignore

REPORT_PATH = Path("evaluation_report.md")

# The report was already written to evaluation_report.md by generate_report()
if REPORT_PATH.exists():
    print(f"📄 Report saved at: {REPORT_PATH.resolve()}")
    print(f"   Size: {REPORT_PATH.stat().st_size:,} bytes")

    # Download to local machine
    print("\nDownloading evaluation_report.md to your local machine...")
    colab_files.download(str(REPORT_PATH))

    # Also save RAGAS DataFrame as CSV
    csv_path = Path("data/ragas_scores.csv")
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    ragas_df.to_csv(csv_path, index=False)
    colab_files.download(str(csv_path))
    print("✅ Downloaded evaluation_report.md and ragas_scores.csv")
else:
    print("⚠️  Report file not found. Check for errors in Cell 5.")

# Print the report inline too
print("\n" + "=" * 70)
print(report_md)